In [4]:
import warnings
warnings.filterwarnings("ignore")
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import albumentations as A
import os
import torch
import pandas as pd

from going_modular.dataloader.multitask import create_concatv2_multitask_dataloader
from going_modular.model.MTLFaceRecognition import MTLFaceRecognition
from going_modular.model.ConcatMTLFaceRecognition import ConcatMTLFaceRecognitionV2
from going_modular.loss.ConcatMultiTaskLoss import ConcatMultiTaskLoss
from going_modular.train_eval.concat_train import fit
from going_modular.utils.transforms import RandomResizedCropRect, GaussianNoise
from going_modular.utils.MultiMetricEarlyStopping import MultiMetricEarlyStopping
from going_modular.utils.ModelCheckPoint import ModelCheckpoint

device = "cuda" if torch.cuda.is_available() else "cpu"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Đặt seed toàn cục
seed = 42
torch.manual_seed(seed)

# --- TÊN THÍ NGHIỆM ---
CONFIGURATION = {
    'experiment_dir': "./experiments/Concat_ALBEDO_NORMAL_PK_SAMPLER_ConvNextV2",
    'checkpoint_1': './Photometric_DB_New/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth',
    'checkpoint_2': './Photometric_DB_New/experiments/Single_NORMALMAP_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth',
    'dataset_dir': 'Photometric_DB_New',
    'type_1': 'albedo',
    'type_2': 'normalmap',

    'backbone': 'convnextv2_tiny',
    'use_sampler': True,

    'device': device,
    'epochs': 119,
    'num_workers': 4,
    'batch_size': 64,
    'image_size': 112,
    'base_lr': 1e-4,
    'num_classes': None,

    # Trọng số loss
    'loss_gender_weight': 1.0,      'loss_da_gender_weight': 0.1,
    'loss_emotion_weight': 0.5,     'loss_da_emotion_weight': 0.1,
    'loss_pose_weight': 1.0,        'loss_da_pose_weight': 0.1,
    'loss_spectacles_weight': 0.5,  'loss_da_spectacles_weight': 0.1,
    'loss_facial_hair_weight': 0.5, 'loss_da_facial_hair_weight': 0.1,
}


# --- 1. Tính num class ---
train_csv_path = os.path.join(CONFIGURATION['dataset_dir'], 'train_set.csv')
df_temp = pd.read_csv(train_csv_path)
CONFIGURATION['num_classes'] = df_temp['id'].max() + 1

# --- 2. Định nghĩa albumentation ---
train_transform = A.Compose([
    RandomResizedCropRect(CONFIGURATION['image_size']),
    GaussianNoise(p=0.2),
    A.HorizontalFlip(p=0.5)
], additional_targets={'albedo': 'image', 'depthmap': 'image', 'normal': 'image'})

test_transform = A.Compose([
    A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size'])
], additional_targets={'albedo': 'image', 'depthmap': 'image', 'normal': 'image'})

In [5]:
# Load Data
train_dataloader, test_dataloader = create_concatv2_multitask_dataloader(CONFIGURATION, train_transform, test_transform)

# --- 3. MODEL SETUP ---
# Load Backbone 1 (Albedo)
print(f"Loading Checkpoint 1: {CONFIGURATION['checkpoint_1']}")
checkpoint_1 = torch.load(CONFIGURATION['checkpoint_1'], map_location=device, weights_only=False)

mtl_backbone1 = MTLFaceRecognition(CONFIGURATION['backbone'], CONFIGURATION['num_classes'])
state_dict_1 = {k.replace('module.', ''): v for k, v in checkpoint_1['model_state_dict'].items()}
mtl_backbone1.load_state_dict(state_dict_1, strict=False)

# Load Backbone 2 (Normal)
print(f"Loading Checkpoint 2: {CONFIGURATION['checkpoint_2']}")
checkpoint_2 = torch.load(CONFIGURATION['checkpoint_2'], map_location=device, weights_only=False)

mtl_backbone2 = MTLFaceRecognition(CONFIGURATION['backbone'], CONFIGURATION['num_classes'])
state_dict_2 = {k.replace('module.', ''): v for k, v in checkpoint_2['model_state_dict'].items()}
mtl_backbone2.load_state_dict(state_dict_2, strict=False)

# Combine Model
model = ConcatMTLFaceRecognitionV2(mtl_backbone1, mtl_backbone2, CONFIGURATION['num_classes'])

# Freeze Backbones
for param in model.mtl_backbone1.parameters(): param.requires_grad = False
for param in model.mtl_backbone2.parameters(): param.requires_grad = False

criterion = ConcatMultiTaskLoss(train_csv_path, CONFIGURATION)
optimizer = Adam(model.parameters(), lr=CONFIGURATION['base_lr'])
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=40, T_mult=1, eta_min=1e-6)

# --- 4. CHECKPOINT & EARLY STOP ---
model_checkpoint = ModelCheckpoint(
    output_dir=CONFIGURATION['experiment_dir'],
    save_freq=5,
    verbose=1
)

early_stopping = MultiMetricEarlyStopping(
    monitor_keys=['cosine_auc'], patience=1500, mode='max', verbose=0,
    save_dir=CONFIGURATION['experiment_dir'], start_from_epoch=0
)

# --- 5. FIT ---
fit(
    conf=CONFIGURATION,
    start_epoch=0,
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    early_stopping=early_stopping,
    model_checkpoint=model_checkpoint,
    existing_manager=early_stopping
)

>>> SingleLoader: MODE = PK SAMPLER
Loading Checkpoint 1: ./Photometric_DB_New/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth


FileNotFoundError: [Errno 2] No such file or directory: './Photometric_DB_New/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth'